## Notebook 2: Consumer Choice Modeling — Hydroponic and Organic Lettuce

This notebook examines how hydroponic production, organic labelling, and price are associated with consumers’ lettuce choices. Using the cleaned dataset prepared in Notebook 1, the analysis will apply an appropriate choice model that accounts for repeated choices made by the same participants. The findings will help explain consumer preferences for different lettuce production attributes and provide evidence relevant to sustainable agricultural production and marketing.

In [ ]:
# Load the cleaned consumer-choice dataset
from google.colab import files
import pandas as pd

uploaded = files.upload()
file_name = next(iter(uploaded))
choice_df = pd.read_csv(file_name)

print("Dataset dimensions:", choice_df.shape)
print("Participants:", choice_df["ID"].nunique())
print("Choice sets:", choice_df[["ID", "cset"]].drop_duplicates().shape[0])
print("\nFirst five rows:")
display(choice_df.head())

Saving lettuce_consumer_choice_clean.csv to lettuce_consumer_choice_clean.csv
Dataset dimensions: (4752, 7)
Participants: 198
Choice sets: 1584

First five rows:


,ID,cset,Alternative,Choice,Hydro,Organic,Price
0,1,1,1,1,1,1,2
1,1,1,2,0,1,0,3
2,1,1,3,0,0,0,0
3,1,2,1,0,0,0,4
4,1,2,2,1,1,0,1


In [ ]:
# Validate choice outcomes and explanatory-variable variation
print("Choice outcome counts:")
print(choice_df["Choice"].value_counts().sort_index())

print("\nUnique values of explanatory variables:")
for col in ["Hydro", "Organic", "Price"]:
    print(f"{col}: {sorted(choice_df[col].unique())}")

print("\nSelected alternatives per choice set:")
print(
    choice_df.groupby(["ID", "cset"])["Choice"]
    .sum()
    .value_counts()
    .sort_index()
)

Choice outcome counts:
Choice
0    3168
1    1584
Name: count, dtype: int64

Unique values of explanatory variables:
Hydro: [np.int64(0), np.int64(1)]
Organic: [np.int64(0), np.int64(1)]
Price: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Selected alternatives per choice set:
Choice
1    1584
Name: count, dtype: int64


In [ ]:
# Check price values for product and no-purchase alternatives
print("Price distribution by alternative:")
print(pd.crosstab(choice_df["Alternative"], choice_df["Price"]))

print("\nPrice range for lettuce alternatives:")
print(
    choice_df.loc[choice_df["Alternative"].isin([1, 2]), "Price"]
    .value_counts()
    .sort_index()
)

Price distribution by alternative:
Price           0    1    2    3    4
Alternative                          
1               0  396  396  396  396
2               0  396  396  396  396
3            1584    0    0    0    0

Price range for lettuce alternatives:
Price
1    792
2    792
3    792
4    792
Name: count, dtype: int64


In [ ]:
# Install the package used for conditional logistic choice modelling
!pip -q install statsmodels

In [ ]:
# Prepare and validate the choice-model dataset
choice_df = choice_df.copy()
choice_df["ChoiceSet"] = (
    choice_df["ID"].astype(str) + "_" +
    choice_df["cset"].astype(str)
)

# Keep the two lettuce attributes and price for the initial model
product_rows = choice_df[choice_df["Alternative"].isin([1, 2])].copy()

print("Choice sets:", choice_df["ChoiceSet"].nunique())
print("Selected alternatives per set:")
print(choice_df.groupby("ChoiceSet")["Choice"].sum().value_counts().sort_index())

print("\nSelected product rows:", len(product_rows))
print("Selected lettuce alternatives:", int(product_rows["Choice"].sum()))
print("No-purchase selections:", int(choice_df.loc[choice_df["Alternative"] == 3, "Choice"].sum()))

Choice sets: 1584
Selected alternatives per set:
Choice
1    1584
Name: count, dtype: int64

Selected product rows: 3168
Selected lettuce alternatives: 1153
No-purchase selections: 431


In [ ]:
# Import the conditional logit model
import statsmodels
from statsmodels.discrete.conditional_models import ConditionalLogit

print("Statsmodels version:", statsmodels.__version__)
print("ConditionalLogit is available.")

Statsmodels version: 0.15.0
ConditionalLogit is available.


In [ ]:
# Estimate an initial conditional logit choice model
import numpy as np
from statsmodels.discrete.conditional_models import ConditionalLogit

model_df = choice_df.copy()
model_df["Product"] = (model_df["Alternative"] != 3).astype(int)

predictors = ["Product", "Hydro", "Organic", "Price"]

model = ConditionalLogit(
    endog=model_df["Choice"].astype(int),
    exog=model_df[predictors].astype(float),
    groups=model_df["ChoiceSet"]
)

result = model.fit(method="bfgs", maxiter=200, disp=False)

print(result.summary())

                  Conditional Logit Model Regression Results                  
Dep. Variable:                 Choice   No. Observations:                 4752
Model:               ConditionalLogit   No. groups:                       1584
Log-Likelihood:               -1481.5   Min group size:                      3
Method:                          bfgs   Max group size:                      3
Date:                Sun, 25 Oct 2026   Mean group size:                   3.0
Time:                        10:25:33                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Product        1.2254      0.108     11.386      0.000       1.014       1.436
Hydro         -0.1607      0.085     -1.893      0.058      -0.327       0.006
Organic        0.8640      0.077     11.233      0.000       0.713       1.015
Price         -0.6043      0.035    -17.285      0.0

In [ ]:
# Bootstrap participants to account for repeated choices
import warnings
import numpy as np
import pandas as pd
from statsmodels.discrete.conditional_models import ConditionalLogit

predictors = ["Product", "Hydro", "Organic", "Price"]
participant_ids = model_df["ID"].unique()
rng = np.random.default_rng(42)
bootstrap_coefs = []

for b in range(200):
    sampled_ids = rng.choice(
        participant_ids, size=len(participant_ids), replace=True
    )
    parts = []

    for draw_id, participant_id in enumerate(sampled_ids):
        part = model_df[model_df["ID"] == participant_id].copy()
        part["BootChoiceSet"] = (
            str(draw_id) + "_" + part["cset"].astype(str)
        )
        parts.append(part)

    boot_df = pd.concat(parts, ignore_index=True)

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            boot_model = ConditionalLogit(
                endog=boot_df["Choice"].astype(int),
                exog=boot_df[predictors].astype(float),
                groups=boot_df["BootChoiceSet"]
            )
            boot_result = boot_model.fit(
                method="bfgs", maxiter=200, disp=False
            )

        if np.all(np.isfinite(boot_result.params)):
            bootstrap_coefs.append(boot_result.params)

    except Exception:
        continue

boot_results = pd.DataFrame(bootstrap_coefs)

print(f"Successful bootstrap replications: {len(boot_results)} of 200")

if len(boot_results) >= 100:
    bootstrap_summary = pd.DataFrame({
        "Original_Coefficient": result.params,
        "Bootstrap_SE": boot_results.std(),
        "CI_Lower_95": boot_results.quantile(0.025),
        "CI_Upper_95": boot_results.quantile(0.975)
    })
    print("\nParticipant-bootstrap results:")
    print(bootstrap_summary.round(3).to_string())
else:
    print("Too few successful replications. Do not interpret the intervals yet.")

Successful bootstrap replications: 200 of 200

Participant-bootstrap results:
         Original_Coefficient  Bootstrap_SE  CI_Lower_95  CI_Upper_95
Product                 1.225         0.190        0.899        1.602
Hydro                  -0.161         0.100       -0.328        0.046
Organic                 0.864         0.087        0.680        1.023
Price                  -0.604         0.046       -0.706       -0.528


In [9]:
# Convert model coefficients and bootstrap intervals to odds ratios
odds_summary = pd.DataFrame({
    "Coefficient": result.params,
    "Odds_Ratio": np.exp(result.params),
    "OR_CI_Lower_95": np.exp(boot_results.quantile(0.025)),
    "OR_CI_Upper_95": np.exp(boot_results.quantile(0.975))
})

print(odds_summary.round(3).to_string())

         Coefficient  Odds_Ratio  OR_CI_Lower_95  OR_CI_Upper_95
Product        1.225       3.405           2.458           4.961
Hydro         -0.161       0.852           0.720           1.047
Organic        0.864       2.373           1.974           2.780
Price         -0.604       0.546           0.494           0.590


In [10]:
# Create a compact results table for the research report
final_results = odds_summary.copy()
final_results.index.name = "Variable"
final_results = final_results.reset_index()

final_results.to_csv("lettuce_choice_model_results.csv", index=False)

print("Saved: lettuce_choice_model_results.csv")
display(final_results.round(3))

Saved: lettuce_choice_model_results.csv


,Variable,Coefficient,Odds_Ratio,OR_CI_Lower_95,OR_CI_Upper_95
0,Product,1.225,3.405,2.458,4.961
1,Hydro,-0.161,0.852,0.720,1.047
2,Organic,0.864,2.373,1.974,2.780
3,Price,-0.604,0.546,0.494,0.590
